# Kaggle Research — Final Wrapper
Upload this notebook + your saved models as a Kaggle Dataset, then run on Kaggle GPUs.

## 1. Load saved models
Models are stored in the `models/` directory after `main.py --export-final`.

In [ ]:
import os, glob, json, pickle
import pandas as pd
import numpy as np

MODEL_DIR = "/kaggle/input/kaggle-research-models/models"
models = []
for pkl in glob.glob(os.path.join(MODEL_DIR, "*.pkl")):
    with open(pkl, "rb") as f:
        models.append(pickle.load(f))
print(f"Loaded {len(models)} models")

## 2. Load test data & predict

In [ ]:
test = pd.read_csv("/kaggle/input/competition/test.csv")
ids = test["id"]
X_test = test.drop(columns=["id", "target"] if "target" in test.columns else ["id"])
X_test = X_test.select_dtypes(include=[np.number]).fillna(-1).values

preds = np.column_stack([m.predict_proba(X_test)[:, 1] for m in models])
ensemble = preds.mean(axis=1)

## 3. Save & submit

In [ ]:
sub = pd.DataFrame({"id": ids, "target": ensemble})
sub.to_csv("submission.csv", index=False)
print(sub.head())